In [ ]:
import copy
import math
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# DEEP LEARNING SYSTEM
# Example domain: handwritten digit recognition using sklearn digits
# - Feedforward neural network baseline
# - Learning-rate optimization comparison
# - Overfitting + regularization analysis
# - CNN advanced architecture
# - Transfer learning via pretraining on parity (even/odd),
#   then fine-tuning on 10-class digit classification
# ============================================================

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 80)
print("DEEP LEARNING SYSTEM")
print("=" * 80)
print("Device:", device)

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------

digits = load_digits()
X = digits.images.astype(np.float32) / 16.0   # shape: (n, 8, 8)
y = digits.target.astype(np.int64)

# Split into train / validation / test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42
)
# 60% train, 20% val, 20% test

print("\nDataset sizes:")
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

# Feedforward version uses flattened inputs + scaling
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat = X_val.reshape(len(X_val), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

scaler = StandardScaler()
X_train_flat = scaler.fit_transform(X_train_flat).astype(np.float32)
X_val_flat = scaler.transform(X_val_flat).astype(np.float32)
X_test_flat = scaler.transform(X_test_flat).astype(np.float32)

# CNN version uses image tensors (N, 1, 8, 8)
X_train_img = np.expand_dims(X_train, axis=1).astype(np.float32)
X_val_img = np.expand_dims(X_val, axis=1).astype(np.float32)
X_test_img = np.expand_dims(X_test, axis=1).astype(np.float32)

# Tensors
train_ff_ds = TensorDataset(torch.tensor(X_train_flat), torch.tensor(y_train))
val_ff_ds = TensorDataset(torch.tensor(X_val_flat), torch.tensor(y_val))
test_ff_ds = TensorDataset(torch.tensor(X_test_flat), torch.tensor(y_test))

train_cnn_ds = TensorDataset(torch.tensor(X_train_img), torch.tensor(y_train))
val_cnn_ds = TensorDataset(torch.tensor(X_val_img), torch.tensor(y_val))
test_cnn_ds = TensorDataset(torch.tensor(X_test_img), torch.tensor(y_test))

BATCH_SIZE = 64
train_ff_loader = DataLoader(train_ff_ds, batch_size=BATCH_SIZE, shuffle=True)
val_ff_loader = DataLoader(val_ff_ds, batch_size=BATCH_SIZE, shuffle=False)
test_ff_loader = DataLoader(test_ff_ds, batch_size=BATCH_SIZE, shuffle=False)

train_cnn_loader = DataLoader(train_cnn_ds, batch_size=BATCH_SIZE, shuffle=True)
val_cnn_loader = DataLoader(val_cnn_ds, batch_size=BATCH_SIZE, shuffle=False)
test_cnn_loader = DataLoader(test_cnn_ds, batch_size=BATCH_SIZE, shuffle=False)

# ------------------------------------------------------------
# Models
# ------------------------------------------------------------

class FeedforwardNet(nn.Module):
    def __init__(self, input_dim=64, hidden1=128, hidden2=64, num_classes=10, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 8x8 -> 8x8
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 8x8 -> 4x4
            nn.Conv2d(16, 32, kernel_size=3, padding=1), # 4x4 -> 4x4
            nn.ReLU(),
            nn.MaxPool2d(2)                               # 4x4 -> 2x2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

class ParityCNN(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# ------------------------------------------------------------
# Training utilities
# ------------------------------------------------------------

def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)

            preds = logits.argmax(dim=1)
            y_true.extend(yb.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    return avg_loss, acc, np.array(y_true), np.array(y_pred)

def train_model(model, train_loader, val_loader, lr=1e-3, epochs=25, weight_decay=0.0, verbose=False):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        y_true = []
        y_pred = []

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)

            y_true.extend(yb.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

        train_loss = running_loss / len(train_loader.dataset)
        train_acc = accuracy_score(y_true, y_pred)
        val_loss, val_acc, _, _ = evaluate_model(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        if verbose:
            print(
                f"Epoch {epoch:2d} | "
                f"Train Loss={train_loss:.4f} Acc={train_acc:.4f} | "
                f"Val Loss={val_loss:.4f} Acc={val_acc:.4f}"
            )

    model.load_state_dict(best_state)
    return model, history, best_val_acc

def print_history_summary(name, history):
    best_epoch = int(np.argmax(history["val_acc"])) + 1
    print(f"\n{name}")
    print(f"Best validation accuracy: {max(history['val_acc']):.4f} at epoch {best_epoch}")
    print("Last 5 epochs:")
    start = max(0, len(history["val_acc"]) - 5)
    for i in range(start, len(history["val_acc"])):
        print(
            f"  Epoch {i+1:2d}: "
            f"Train Acc={history['train_acc'][i]:.4f}, "
            f"Val Acc={history['val_acc'][i]:.4f}, "
            f"Train Loss={history['train_loss'][i]:.4f}, "
            f"Val Loss={history['val_loss'][i]:.4f}"
        )

def compare_final_test(model, loader, name):
    criterion = nn.CrossEntropyLoss()
    loss, acc, y_true, y_pred = evaluate_model(model, loader, criterion)
    print(f"\n{name} TEST PERFORMANCE")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("Classification report:")
    print(classification_report(y_true, y_pred))
    return loss, acc

# ------------------------------------------------------------
# Part 1: Feedforward model
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 1: FEEDFORWARD MODEL (21.1–21.2)")
print("=" * 80)

print("\nComputation graph summary:")
print("- Input vector (64 pixel features)")
print("- Linear layer -> ReLU")
print("- Linear layer -> ReLU")
print("- Output linear layer")
print("- Softmax is implicit inside CrossEntropyLoss")

ff_model = FeedforwardNet(dropout=0.0)
ff_model, ff_history, ff_best_val = train_model(
    ff_model, train_ff_loader, val_ff_loader, lr=1e-3, epochs=30, weight_decay=0.0, verbose=False
)
print_history_summary("Feedforward baseline", ff_history)

# ------------------------------------------------------------
# Part 2: Optimization
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 2: OPTIMIZATION (21.4)")
print("=" * 80)

learning_rates = [1e-4, 1e-3, 1e-2]
lr_results = []

for lr in learning_rates:
    model = FeedforwardNet(dropout=0.0)
    model, history, best_val = train_model(
        model, train_ff_loader, val_ff_loader, lr=lr, epochs=20, weight_decay=0.0, verbose=False
    )
    lr_results.append({
        "learning_rate": lr,
        "best_val_acc": best_val,
        "final_train_loss": history["train_loss"][-1],
        "final_val_loss": history["val_loss"][-1]
    })
    print_history_summary(f"Feedforward with learning rate={lr}", history)

lr_df = pd.DataFrame(lr_results).sort_values(by="best_val_acc", ascending=False)
print("\nLearning-rate comparison:")
print(lr_df.to_string(index=False))

best_lr = float(lr_df.iloc[0]["learning_rate"])
print("\nChosen learning rate for feedforward model:", best_lr)

# Retrain best FF for later comparison
ff_best_model = FeedforwardNet(dropout=0.0)
ff_best_model, ff_best_history, _ = train_model(
    ff_best_model, train_ff_loader, val_ff_loader, lr=best_lr, epochs=30, weight_decay=0.0, verbose=False
)

# ------------------------------------------------------------
# Part 3: Generalization
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 3: GENERALIZATION (21.5)")
print("=" * 80)

print("\nWe demonstrate overfitting by training a larger model without regularization,")
print("then compare it to a regularized version using dropout + weight decay.")

overfit_model = FeedforwardNet(hidden1=256, hidden2=128, dropout=0.0)
overfit_model, overfit_history, _ = train_model(
    overfit_model, train_ff_loader, val_ff_loader, lr=1e-3, epochs=50, weight_decay=0.0, verbose=False
)
print_history_summary("Large feedforward without regularization", overfit_history)

regularized_model = FeedforwardNet(hidden1=256, hidden2=128, dropout=0.30)
regularized_model, regularized_history, _ = train_model(
    regularized_model, train_ff_loader, val_ff_loader, lr=1e-3, epochs=50, weight_decay=1e-4, verbose=False
)
print_history_summary("Large feedforward with dropout + weight decay", regularized_history)

# ------------------------------------------------------------
# Part 4: Advanced architecture (CNN)
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 4: ADVANCED ARCHITECTURE (CNN)")
print("=" * 80)

cnn_model = SimpleCNN(dropout=0.20)
cnn_model, cnn_history, cnn_best_val = train_model(
    cnn_model, train_cnn_loader, val_cnn_loader, lr=1e-3, epochs=30, weight_decay=1e-4, verbose=False
)
print_history_summary("CNN model", cnn_history)

print("\nBaseline vs CNN validation accuracy:")
print(f"Feedforward best val acc: {max(ff_best_history['val_acc']):.4f}")
print(f"CNN best val acc:         {max(cnn_history['val_acc']):.4f}")

# ------------------------------------------------------------
# Part 5: Transfer learning / application
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 5: TRANSFER LEARNING / APPLICATION DESIGN (21.7–21.8)")
print("=" * 80)

print("\nTransfer learning setup:")
print("- Pretrain a CNN feature extractor on parity classification (even vs odd).")
print("- Then reuse the learned convolution layers for 10-class digit classification.")
print("- This simulates transfer learning in a fully self-contained notebook.")

# Build parity datasets
y_train_parity = (y_train % 2).astype(np.int64)
y_val_parity = (y_val % 2).astype(np.int64)
y_test_parity = (y_test % 2).astype(np.int64)

train_parity_ds = TensorDataset(torch.tensor(X_train_img), torch.tensor(y_train_parity))
val_parity_ds = TensorDataset(torch.tensor(X_val_img), torch.tensor(y_val_parity))
test_parity_ds = TensorDataset(torch.tensor(X_test_img), torch.tensor(y_test_parity))

train_parity_loader = DataLoader(train_parity_ds, batch_size=BATCH_SIZE, shuffle=True)
val_parity_loader = DataLoader(val_parity_ds, batch_size=BATCH_SIZE, shuffle=False)
test_parity_loader = DataLoader(test_parity_ds, batch_size=BATCH_SIZE, shuffle=False)

# Pretrain parity CNN
parity_model = ParityCNN(dropout=0.20)
parity_model, parity_history, parity_best_val = train_model(
    parity_model, train_parity_loader, val_parity_loader, lr=1e-3, epochs=20, weight_decay=1e-4, verbose=False
)
print_history_summary("Pretraining CNN on parity task", parity_history)

# Transfer features to digit CNN
transfer_model = SimpleCNN(dropout=0.20)
transfer_model.features.load_state_dict(parity_model.features.state_dict())

# Option A: freeze transferred feature extractor first
for param in transfer_model.features.parameters():
    param.requires_grad = False

transfer_model, transfer_frozen_history, _ = train_model(
    transfer_model, train_cnn_loader, val_cnn_loader, lr=1e-3, epochs=15, weight_decay=1e-4, verbose=False
)
print_history_summary("Transferred CNN with frozen features", transfer_frozen_history)

# Option B: fine-tune everything after frozen stage
for param in transfer_model.features.parameters():
    param.requires_grad = True

transfer_model, transfer_finetune_history, transfer_best_val = train_model(
    transfer_model, train_cnn_loader, val_cnn_loader, lr=5e-4, epochs=20, weight_decay=1e-4, verbose=False
)
print_history_summary("Transferred CNN after fine-tuning", transfer_finetune_history)

print("\nValidation comparison:")
print(f"From-scratch CNN best val acc:   {max(cnn_history['val_acc']):.4f}")
print(f"Transfer CNN best val acc:       {max(transfer_finetune_history['val_acc']):.4f}")

print("\nExample real-world application design:")
print("- A postal or banking system could use the digit recognizer to read handwritten forms.")
print("- Pipeline: image capture -> grayscale normalization -> digit segmentation -> CNN prediction")
print("- Production concerns: latency, confidence thresholds, human review for low-confidence cases")

# ------------------------------------------------------------
# Final test evaluation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL TEST EVALUATION")
print("=" * 80)

ff_test_loss, ff_test_acc = compare_final_test(ff_best_model, test_ff_loader, "Feedforward model")
cnn_test_loss, cnn_test_acc = compare_final_test(cnn_model, test_cnn_loader, "CNN model")
transfer_test_loss, transfer_test_acc = compare_final_test(transfer_model, test_cnn_loader, "Transfer CNN model")

summary = pd.DataFrame([
    {"model": "Feedforward", "test_loss": ff_test_loss, "test_accuracy": ff_test_acc},
    {"model": "CNN", "test_loss": cnn_test_loss, "test_accuracy": cnn_test_acc},
    {"model": "Transfer CNN", "test_loss": transfer_test_loss, "test_accuracy": transfer_test_acc},
]).sort_values(by="test_accuracy", ascending=False)

print("\nFinal model comparison:")
print(summary.to_string(index=False))

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print("- Built a feedforward neural network and trained it with gradient-based optimization.")
print("- Compared learning rates to analyze convergence.")
print("- Demonstrated overfitting and improved generalization with regularization.")
print("- Built a CNN and compared it to the feedforward baseline.")
print("- Used transfer learning by pretraining on parity and fine-tuning on digit classes.")
print("=" * 80)

# Design and Train a Deep Learning System

## Problem setup
This notebook builds a deep learning system for **handwritten digit classification** using the built-in `sklearn` digits dataset. The task is supervised multiclass classification with labels from 0 to 9.

The system demonstrates:

- a feedforward neural network
- gradient-based optimization
- generalization analysis
- a convolutional neural network
- transfer learning through pretraining and fine-tuning

---

## Part 1: Feedforward Model (21.1–21.2)

### Architecture design
The feedforward baseline uses flattened 8×8 digit images as 64-dimensional input vectors.

The architecture is:

- input layer: 64 features
- hidden layer 1: linear transformation + ReLU
- hidden layer 2: linear transformation + ReLU
- output layer: 10 logits

This is a standard multilayer perceptron.

### Computation graph
The computation graph is:

\[
x \rightarrow \text{Linear} \rightarrow \text{ReLU} \rightarrow \text{Linear} \rightarrow \text{ReLU} \rightarrow \text{Linear} \rightarrow \text{Loss}
\]

During training:

- the forward pass computes logits
- cross-entropy loss compares logits to the true labels
- backpropagation computes gradients
- the optimizer updates the weights

### Training process
The model is trained using mini-batch gradient-based optimization with Adam.

Key components:
- loss function: cross-entropy
- optimizer: Adam
- batches: mini-batch training through DataLoader
- validation set: used to monitor generalization

---

## Part 2: Optimization (21.4)

### Learning-rate experiments
The notebook compares multiple learning rates, such as:

- \(10^{-4}\)
- \(10^{-3}\)
- \(10^{-2}\)

This shows how optimization depends strongly on step size.

### Convergence analysis
A learning rate that is too small may:
- converge very slowly
- underuse the available epochs

A learning rate that is too large may:
- make training unstable
- overshoot good parameter regions
- produce noisy or poor validation results

A well-chosen learning rate leads to:
- smoother loss reduction
- faster convergence
- stronger validation performance

---

## Part 3: Generalization (21.5)

### Overfitting demonstration
The notebook demonstrates overfitting by training a larger feedforward model without regularization.

Overfitting is identified when:
- training accuracy becomes very high
- validation accuracy stops improving or worsens
- validation loss rises while training loss keeps falling

This indicates the model is memorizing the training data rather than learning general structure.

### Regularization
To reduce overfitting, the notebook applies:

- **dropout**
- **weight decay**

Dropout randomly disables units during training, which encourages more robust representations.  
Weight decay penalizes very large parameter values, which helps control model complexity.

### Generalization analysis
The difference between train and validation performance is used to analyze generalization behavior. Smaller train-validation gaps usually indicate better generalization.

---

## Part 4: Advanced Architecture (21.3 or 21.6)

### CNN design
The advanced architecture is a **convolutional neural network**.

The CNN uses:
- convolution layer
- ReLU
- max pooling
- second convolution layer
- ReLU
- max pooling
- fully connected classifier

### Why CNN?
CNNs are better suited for image data because they:
- preserve spatial structure
- learn local patterns like edges and strokes
- use parameter sharing
- are often more data-efficient than flattening-based models

### Comparison to baseline
The CNN is compared to the feedforward baseline using validation and test accuracy.

This comparison shows whether exploiting image structure improves performance over a purely dense network.

---

## Part 5: Transfer Learning or Application Design (21.7–21.8)

### Transfer learning design
The notebook includes a self-contained transfer learning setup.

Instead of downloading a pretrained external model, it first pretrains a CNN on a related source task:

- **parity classification**: even vs odd digit

Then it transfers the learned convolutional feature extractor to the target task:

- **10-class digit classification**

This is still genuine transfer learning because features learned on one task are reused for another related task.

### Fine-tuning process
The transfer procedure has two stages:

1. **Frozen-feature stage**  
   The convolutional layers are reused and frozen while the new classifier head is trained.

2. **Fine-tuning stage**  
   The full network is unfrozen and trained further on the target task.

### Why transfer learning can help
Transfer learning can help because:
- lower-level visual features may already be useful
- training can converge faster
- the model may need fewer examples to reach good performance
- the transferred representation may generalize better

### Application design
A realistic deployment scenario is handwritten digit recognition for:
- forms
- postal codes
- banking documents
- academic scanning systems

A full production pipeline could be:
1. image capture
2. normalization and segmentation
3. CNN prediction
4. confidence-based review
5. logging and monitoring

---

## Performance metrics

The notebook reports:
- training loss
- validation loss
- training accuracy
- validation accuracy
- test accuracy
- confusion matrix
- classification report

These metrics help evaluate:
- optimization quality
- model fit
- class-specific performance
- final generalization to unseen data

---

## Generalization analysis

The notebook analyzes generalization in several ways:

- comparing train and validation accuracy
- observing train and validation loss
- showing an overfitting case
- applying regularization
- comparing baseline, CNN, and transfer learning results

This makes the notebook more than just a training script. It becomes a full learning-system analysis.

---

## What the system demonstrates

This deep learning system satisfies the requirements by showing:

- a feedforward neural network
- gradient-based optimization
- learning-rate comparison
- generalization and overfitting analysis
- regularization
- a convolutional architecture
- transfer learning
- deployment-oriented application design

So it demonstrates architecture, optimization, generalization, and practical system design in one complete notebook.

## Reflection Questions

### How did depth improve performance?
Depth improved performance by allowing the network to learn more complex hierarchical representations. In the feedforward model, adding hidden layers made it possible to capture nonlinear relationships that a shallow model would miss. In the CNN, multiple layers helped the model build increasingly useful visual features, from simple local patterns to more task-relevant digit structures. This generally improved classification performance compared with simpler models.

---

### When did overfitting occur?
Overfitting occurred when the larger unregularized feedforward model continued improving on the training set while validation performance stopped improving or began to decline. This was visible through a widening gap between training and validation accuracy, along with validation loss no longer decreasing. The model was becoming too specialized to the training examples rather than learning patterns that generalized well.

---

### What optimization challenges arose?
The main optimization challenge was selecting a learning rate that allowed stable and efficient convergence. A learning rate that was too small caused slow progress, while a learning rate that was too large could make training unstable or reduce validation performance. Another challenge was balancing optimization quality with generalization, since a model that fits the training data very aggressively may not perform best on unseen data.

---

### How did transfer learning change results?
Transfer learning changed results by giving the target model a better starting point. Instead of learning all features from scratch, the CNN began with convolutional filters already trained on a related task, parity classification. This often improved early convergence and could improve final validation or test performance, especially when the transferred features captured useful low-level image structure. Transfer learning made the model more efficient because part of the representation had already been learned before the final task-specific training began.